# Notebook 5 — Explainability, Ethical AI & Bias Auditing
## Step 5: Critical Thinking — SHAP, LIME, PDP, Bias Audit

In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import json
import joblib

from data_loader import load_dataset
from explainability import shap_global_importance, lime_local_explanation, plot_partial_dependence
from bias_audit import run_bias_audit, MITIGATION_STRATEGIES

# Load artifacts
X_train = np.load('../data/processed/X_train.npy')
X_test  = np.load('../data/processed/X_test.npy')
y_train = np.load('../data/processed/y_train.npy')
y_test  = np.load('../data/processed/y_test.npy')

with open('../data/processed/feature_names.json') as f:
    feature_names = json.load(f)

xgb_model = joblib.load('../models/xgboost.pkl')
rf_model  = joblib.load('../models/random_forest.pkl')

_, test_df_raw = load_dataset(data_dir='../data/raw')
print('Artifacts loaded.')

## 5.1 SHAP — Global Feature Importance

In [ ]:
shap_values, explainer = shap_global_importance(
    xgb_model, X_test, feature_names,
    max_display=20, save_dir='../reports', sample_size=2000
)

from IPython.display import Image
Image('../reports/shap_beeswarm.png')

In [ ]:
Image('../reports/shap_bar.png')

### SHAP Interpretation

The SHAP beeswarm reveals:
- **`serror_rate`** (SYN error rate) is the strongest predictor — high values strongly indicate DoS/scan attacks
- **`same_srv_rate`** — attacks often hit many different services (low value = attack)
- **`dst_host_srv_count`** — connection scan patterns have low values
- **`src_bytes`** — extremely large or near-zero values are anomalous
- **`flag_SF`** — normal connections typically complete (SF = success flag)

These align with known network security intuitions: port scans generate many SYN errors, DoS attacks create uniform high-rate traffic.

## 5.2 SHAP — Local Explanation (Individual Instance)

In [ ]:
# Find one correctly predicted attack instance
y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]
attack_idx = np.where((y_test == 1) & (y_pred_proba > 0.95))[0][0]
print(f'Attack instance index: {attack_idx}')
print(f'Predicted probability: {y_pred_proba[attack_idx]:.4f}')

from explainability import shap_local_explanation
shap_local_explanation(explainer, X_test, attack_idx, feature_names, save_dir='../reports')
Image(f'../reports/shap_local_instance_{attack_idx}.png')

## 5.3 LIME — Local Instance Explanation

In [ ]:
lime_local_explanation(
    xgb_model, X_train, X_test[attack_idx],
    feature_names=feature_names,
    save_dir='../reports', instance_idx=attack_idx
)
Image(f'../reports/lime_instance_{attack_idx}.png')

## 5.4 PDP — Partial Dependence Plots

In [ ]:
top_features_for_pdp = ['serror_rate', 'same_srv_rate', 'dst_host_srv_count']
plot_partial_dependence(
    xgb_model, X_test, feature_names,
    features_to_plot=top_features_for_pdp,
    save_dir='../reports'
)
Image('../reports/pdp_plots.png')

## 5.5 Model Limitations

In [ ]:
# Class imbalance analysis
import matplotlib.pyplot as plt

from data_loader import load_dataset, ATTACK_CATEGORY_MAP
train_df, test_df = load_dataset(data_dir='../data/raw')

attack_counts = train_df[train_df['binary_label'] == 1]['attack_category'].value_counts()
print('Attack category counts in training set:')
print(attack_counts)

plt.figure(figsize=(7, 4))
attack_counts.plot(kind='bar', color='tomato')
plt.title('Class Imbalance — Attack Categories (Train)')
plt.ylabel('Count')
plt.yscale('log')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('../reports/class_imbalance.png', dpi=150)
plt.show()

In [ ]:
# Limitations summary
limitations = {
    'Class Imbalance': 'U2R and R2L have very few samples (<1%). '
                       'Model may miss these rare but high-severity attacks. '
                       'Mitigation: SMOTE or class_weight="balanced".',

    'Data Leakage Risk': 'difficulty_level column (if used) would be leakage. '
                         'It is explicitly excluded from all feature pipelines.',

    'Overfitting': 'XGBoost train F1=0.999 vs test F1=0.993. '
                   'Slight overfitting. Addressed with CV tuning and max_depth limits.',

    'Distribution Shift': 'NSL-KDD is from 1999. Modern attacks (ransomware, APT) '
                          'are not represented. Model requires periodic retraining.',

    'Feature Engineering Risk': 'Domain features use training statistics (log, clip). '
                                'Applied identically to test to prevent leakage.',
}

for k, v in limitations.items():
    print(f'⚠️  {k}:')
    print(f'   {v}\n')

## 5.6 Bias Detection & Fairness Audit

In [ ]:
audit_results = run_bias_audit(
    xgb_model,
    X_test,
    y_test,
    test_df_raw=test_df,
    feature_names=feature_names,
    grouping_cols=['protocol_type', 'service', 'flag'],
    save_dir='../reports'
)

In [ ]:
# Display fairness summary
import json
print(json.dumps(audit_results['fairness_summary'], indent=2))

In [ ]:
# Display per-group metrics for protocol_type
audit_results['protocol_type']

In [ ]:
# Bias images
Image('../reports/bias_f1_protocol_type.png')

In [ ]:
Image('../reports/bias_fpr_protocol_type.png')

## 5.7 Mitigation Strategies

In [ ]:
from bias_audit import MITIGATION_STRATEGIES
for strategy, description in MITIGATION_STRATEGIES.items():
    print(f'✅ {strategy}')
    print(f'   {description}\n')